In [1]:
import zipfile

zip_path = "no_attention.zip"
extract_path = "no_attention"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction terminée ✅")

Extraction terminée ✅


In [2]:
from pathlib import Path
import re
import torch
import pandas as pd

outputs_dir = Path("no_attention/o_attention/outputs")

def epoch_from_name(p: Path):
    m = re.search(r"epoch_(\d+)", p.stem)
    return int(m.group(1)) if m else None

def scalar(x):
    if isinstance(x, torch.Tensor):
        return float(x.item()) if x.numel() == 1 else None
    if isinstance(x, (int, float)):
        return float(x)
    return None

def extract_metrics(organ):
    ckpt_dir = outputs_dir / f"checkpoints_{organ}"
    files = sorted(ckpt_dir.glob("epoch_*.pth"), key=epoch_from_name)

    rows = []
    for f in files:
        c = torch.load(f, map_location="cpu")
        epoch = int(c.get("epoch_1_based", epoch_from_name(f)))
        dice = scalar(c.get("avg_val_dice", c.get("best_val_dice")))
        loss = scalar(c.get("avg_train_loss"))
        rows.append({"organ": organ, "epoch": epoch, "dice": dice, "loss": loss})

    return pd.DataFrame(rows).sort_values("epoch").reset_index(drop=True)

df_liver = extract_metrics("liver")
df_tumor = extract_metrics("tumor")

print("NaN liver:", df_liver[["dice","loss"]].isna().sum().to_dict())
print("NaN tumor:", df_tumor[["dice","loss"]].isna().sum().to_dict())

display(df_liver)
display(df_tumor)

C:\Users\neifa\AppData\Local\Temp\ipykernel_14752\32759811.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  c = torch.load(f, map_location="cpu")
C:\Users\neifa\AppData\

NaN liver: {'dice': 0, 'loss': 0}
NaN tumor: {'dice': 0, 'loss': 0}


,organ,epoch,dice,loss
0,liver,1,0.727308,0.090220
1,liver,2,0.750366,0.013988
2,liver,3,0.815238,0.007698
3,liver,4,0.860124,0.005742
4,liver,5,0.835974,0.004857
5,liver,6,0.867081,0.004354
6,liver,7,0.883763,0.004122
7,liver,8,0.771970,0.003825


,organ,epoch,dice,loss
0,tumor,1,0.772098,0.107315
1,tumor,2,0.769647,0.034907
2,tumor,3,0.598680,0.025636
3,tumor,4,0.783105,0.021369
4,tumor,5,0.774723,0.018299
5,tumor,6,0.803816,0.016534
6,tumor,7,0.603601,0.013993
7,tumor,8,0.795992,0.013159
8,tumor,9,0.755256,0.012013
9,tumor,10,0.821798,0.011280


In [4]:
from pathlib import Path

# Reprend le même dossier de sortie que tes checkpoints
txt_path = Path("no_attention/o_attention/outputs/metrics_liver_tumor_per_epoch.txt")

with open(txt_path, "w", encoding="utf-8") as f:
    f.write("=== LIVER: Dice/Loss par epoch ===\n")
    f.write(df_liver[["epoch", "dice", "loss"]].to_string(index=False))
    f.write("\n\n=== TUMOR: Dice/Loss par epoch ===\n")
    f.write(df_tumor[["epoch", "dice", "loss"]].to_string(index=False))

print(f"Fichier .txt créé: {txt_path}")

Fichier .txt créé: no_attention\o_attention\outputs\metrics_liver_tumor_per_epoch.txt
